# ChatPromptTemplateの高度な機能

## 1、部分変数の事前設定：partial()

例：

In [5]:
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from rich import print
# 元のテンプレート
template = ChatPromptTemplate.from_messages([
    ("system", "あなたは{role}で、対象ユーザーは{audience}です"),
    ("user", "{task}")
])


result1 = template.invoke({"role":"ガイド","audience":"観光客","task":"北京の故宮を紹介してください"})
result2 = template.invoke({"role":"ガイド","audience":"観光客","task":"北京の頤和園を紹介してください"})

print(result1)
print(result2)

ChatPromptValue(
    messages=[
        SystemMessage(
            content='あなたはガイドで、対象ユーザーは観光客です',
            additional_kwargs={},
            response_metadata={}
        ),
        HumanMessage(content='北京の故宮を紹介してください', additional_kwargs={}, response_metadata={})
    ]
)

ChatPromptValue(
    messages=[
        SystemMessage(
            content='あなたはガイドで、対象ユーザーは観光客です',
            additional_kwargs={},
            response_metadata={}
        ),
        HumanMessage(content='北京の頤和園を紹介してください', additional_kwargs={}, response_metadata={})
    ]
)

上記のコードは partial() を使って最適化できます

In [8]:
from langchain_core.prompts import ChatPromptTemplate

# 元のテンプレート
template = ChatPromptTemplate.from_messages([
    ("system", "あなたは{role}で、対象ユーザーは{audience}です"),
    ("user", "{task}")
])

# 部分変数の事前設定
final_template = template.partial(role="ガイド",audience="観光客")


result1 = final_template.invoke({"task":"北京の故宮を紹介してください"})
result2 = final_template.invoke({"task":"北京の頤和園を紹介してください"})

print(result1)
print(result2)

ChatPromptValue(
    messages=[
        SystemMessage(
            content='あなたはガイドで、対象ユーザーは観光客です',
            additional_kwargs={},
            response_metadata={}
        ),
        HumanMessage(content='北京の故宮を紹介してください', additional_kwargs={}, response_metadata={})
    ]
)

ChatPromptValue(
    messages=[
        SystemMessage(
            content='あなたはガイドで、対象ユーザーは観光客です',
            additional_kwargs={},
            response_metadata={}
        ),
        HumanMessage(content='北京の頤和園を紹介してください', additional_kwargs={}, response_metadata={})
    ]
)

例：

In [7]:
# シナリオ：部門ごとの専用テンプレートを作成
base_template = ChatPromptTemplate.from_messages([
    ("system", "あなたは{department}の{role}です"),
    ("user", "{task}")
])

# IT 部門
it_template = base_template.partial(
    department="IT 部門",
    role="テクニカルサポート"
)

# 営業部門
sales_template = base_template.partial(
    department="営業部門",
    role="営業コンサルタント"
)

sales_template.invoke({"task":"なぜ毎年年末に自動車のセールが行われるのですか"})

ChatPromptValue(messages=[SystemMessage(content='あなたは営業部門の営業コンサルタントです', additional_kwargs={}, response_metadata={}), HumanMessage(content='なぜ毎年年末に自動車のセールが行われるのですか', additional_kwargs={}, response_metadata={})])

## 2、メッセージプレースホルダー

### 2.1 placeholderを使用

例

In [6]:
template = ChatPromptTemplate.from_messages([
    ("system","私はAIアシスタントです"),
    ("placeholder","{conversation}")
])

result = template.invoke({
    "conversation" : [
        ("human","こんにちは、明日の天気はどうですか？"),
        ("ai","明日は晴れです"),
        ("human","明後日の天気はどうですか？")
    ]
})

print(result)

ChatPromptValue(
    messages=[
        SystemMessage(content='私はAIアシスタントです', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='こんにちは、明日の天気はどうですか？', additional_kwargs={}, response_metadata={}),
        AIMessage(
            content='明日は晴れです',
            additional_kwargs={},
            response_metadata={},
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(content='明後日の天気はどうですか？', additional_kwargs={}, response_metadata={})
    ]
)

### 2.2 MessagesPlaceholderを使用

例：

In [9]:
from langchain_core.prompts import MessagesPlaceholder
template = ChatPromptTemplate.from_messages([
    ("system","私はAIアシスタントです"),
    MessagesPlaceholder(variable_name="conversation")
])

result = template.invoke({
    "conversation" : [
        ("human","こんにちは、明日の天気はどうですか？"),
        ("ai","明日は晴れです"),
        ("human","明後日の天気はどうですか？")
    ]
})

print(result)

ChatPromptValue(
    messages=[
        SystemMessage(content='私はAIアシスタントです', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='こんにちは、明日の天気はどうですか？', additional_kwargs={}, response_metadata={}),
        AIMessage(
            content='明日は晴れです',
            additional_kwargs={},
            response_metadata={},
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(content='明後日の天気はどうですか？', additional_kwargs={}, response_metadata={})
    ]
)

In [10]:
from langchain_core.messages import AIMessage,HumanMessage

result = template.invoke({
    "conversation" : [
        HumanMessage("こんにちは、明日の天気はどうですか？"),
        AIMessage("明日は晴れです"),
        HumanMessage("明後日の天気はどうですか？"),
    ]
})

print(result)

ChatPromptValue(
    messages=[
        SystemMessage(content='私はAIアシスタントです', additional_kwargs={}, response_metadata={}),
        HumanMessage(content='こんにちは、明日の天気はどうですか？', additional_kwargs={}, response_metadata={}),
        AIMessage(
            content='明日は晴れです',
            additional_kwargs={},
            response_metadata={},
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(content='明後日の天気はどうですか？', additional_kwargs={}, response_metadata={})
    ]
)

例：履歴メッセージの保存

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "あなたはとても親切なAIアシスタントです"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}")
    ]
)

prompt_template.invoke(
    {
        "history": [
            ("human", "5 + 2 = ?"),
            ("ai", "5 + 2 = 7")
        ],
        "question": "結果をさらに4倍するとどうなりますか？"
    }
)

ChatPromptValue(messages=[SystemMessage(content='あなたはとても親切なAIアシスタントです', additional_kwargs={}, response_metadata={}), HumanMessage(content='5 + 2 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='5 + 2 = 7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='結果をさらに4倍するとどうなりますか？', additional_kwargs={}, response_metadata={})])

## 3、再利用可能なテンプレートライブラリ

テンプレートを格納する具体的な py ファイルを定義

In [14]:
from langchain_core.prompts import ChatPromptTemplate

class PromptLibrary:
    """再利用可能なプロンプトテンプレートライブラリ"""

    TRANSLATOR = ChatPromptTemplate.from_messages([
        ("system", "あなたはプロの翻訳者で、{source_lang}と{target_lang}に精通しています"),
        ("user", "以下のテキストを翻訳してください：\n{text}")
    ])

    CODE_REVIEWER = ChatPromptTemplate.from_messages([
        ("system", "あなたは{language}のコードレビュー専門家で、{focus}に重点を置いています"),
        ("user", "コードをレビューしてください：\n```{language}\n{code}\n```")
    ])

    SUMMARIZER = ChatPromptTemplate.from_messages([
        ("system", "あなたはコンテンツ要約の専門家です"),
        ("user", "以下の内容を{num}個の要点にまとめてください：\n{content}")
    ])

    TUTOR = ChatPromptTemplate.from_messages([
        ("system", "あなたは{subject}の講師で、学生のレベルは：{level}です"),
        ("user", "{question}")
    ])

他のファイルから呼び出す：

In [13]:
# from templates import PromptLibrary

messages = PromptLibrary.TRANSLATOR.format_messages(
    source_lang="英語",
    target_lang="中国語",
    text="Hello World"
)